In [1]:
from __future__ import annotations

import os
import re
import json
import time
import logging
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests

In [2]:
# -----------------------
# Config
# -----------------------
ROOT = "https://ensembledata.com/apis"
ENDPOINT = "/threads/user/posts"
TOKEN = os.getenv("ENSEMBLEDATA_TOKEN", "GobJlP7vKjn4Z5Jw")

INPUT_CSV = Path("Threads/user_info.csv")          # <- input
OUT_DIR = Path("Threads") / "post_info"    # <- output folder

CHUNK_SIZE = 5
SLEEP_SECS = 0.35
TIMEOUT_SECS = 60

OVERWRITE_EXISTING = False
MAX_PAGES_PER_USER = 9999   # safety

In [4]:
df=pd.read_csv(INPUT_CSV)
df.head()

,query_id,http_status,user_id,username,full_name,is_verified,text_post_app_is_private,follower_count,profile_pic_url,hd_profile_pic_urls,biography,text_biography_plain,bio_links,is_threads_only_user,scrape_ts,source_file
0,1090677444,200,1090677444,nikegirlseybl,Nike Girls EYBL,True,False,12728,https://scontent-iad3-1.cdninstagram.com/v/t51...,https://scontent-iad3-1.cdninstagram.com/v/t51...,Nike Elite Youth Girls Basketball. #2026EYBL,Nike Elite Youth Girls Basketball. #2026EYBL,http://www.nikegirlseybl.com/,False,2026-01-27T14:07:19.947860+00:00,1090677444.json
1,13460080,200,13460080,nike,Nike,True,False,14732908,https://scontent-atl3-3.cdninstagram.com/v/t51...,https://scontent-atl3-3.cdninstagram.com/v/t51...,Just Do It.,Just Do It.,http://empli.fi/nike,False,2026-01-19T23:55:10.230107+00:00,13460080.json
2,20269764,200,20269764,adidas,adidas,True,False,2548288,https://scontent-atl3-1.cdninstagram.com/v/t51...,https://scontent-atl3-1.cdninstagram.com/v/t51...,#YouGotThis,#YouGotThis,https://linktr.ee/adidas,False,2026-01-19T23:55:07.905283+00:00,20269764.json
3,2568684,200,2568684,nikediamond,Nike Diamond,True,False,154986,https://scontent-ord5-1.cdninstagram.com/v/t51...,https://scontent-ord5-1.cdninstagram.com/v/t51...,Winning Isn’t For Everyone. It’s for those who...,Winning Isn’t For Everyone. It’s for those who...,https://www.mlb.com/rbi/program-near-you,False,2026-01-27T14:07:14.830497+00:00,2568684.json
4,306787899,200,306787899,nikebasketball,Nike Basketball,True,False,1632090,https://scontent-sea1-1.cdninstagram.com/v/t51...,https://scontent-sea1-1.cdninstagram.com/v/t51...,Fit for a King. The LeBron XXIII has arrived.,Fit for a King. The LeBron XXIII has arrived.,http://empli.fi/nikebasketball,False,2026-01-27T14:07:11.043129+00:00,306787899.json


In [6]:
# -----------------------
# Helpers
# -----------------------
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def yyyymmdd_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%d")

def safe_filename(name: str, max_len: int = 180) -> str:
    name = str(name).strip()
    name = re.sub(r"[^\w\-\.]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return (name or "unknown")[:max_len]

def safe_get(d: Any, path: List[Any], default=None):
    cur = d
    for k in path:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        elif isinstance(cur, list) and isinstance(k, int) and 0 <= k < len(cur):
            cur = cur[k]
        else:
            return default
    return cur


def find_next_cursor(payload: Dict[str, Any]) -> Optional[str]:
    """
    Your sample: payload["data"] is a list; each element has a "cursor".
    Some APIs return pagination cursor per edge; if present and non-empty,
    we use it for next page.
    """
    data_list = payload.get("data")
    if not isinstance(data_list, list) or not data_list:
        return None

    # Try cursor at top-level edge
    c = data_list[-1].get("cursor") if isinstance(data_list[-1], dict) else None
    if isinstance(c, str) and c.strip():
        return c.strip()

    # Sometimes cursor could exist elsewhere; try a couple fallbacks
    c2 = safe_get(payload, ["paging", "cursor"], default=None)
    if isinstance(c2, str) and c2.strip():
        return c2.strip()

    return None


# -----------------------
# API call
# -----------------------
def scrape_user_posts(user_id: str, token: str, chunk_size: int, cursor: Optional[str] = None) -> Dict[str, Any]:
    url = ROOT + ENDPOINT
    params = {"id": user_id, "chunk_size": chunk_size, "token": token}
    if cursor:
        # EnsembleData often uses cursor/next_cursor style; if your API uses a different param
        # (e.g., "max_id" or "end_cursor"), change the key here accordingly.
        params["cursor"] = cursor

    res = requests.get(url, params=params, timeout=TIMEOUT_SECS)
    res.raise_for_status()
    raw = res.json()
    return raw

In [7]:
# -----------------------
# Step 1: Scrape from CSV and save JSONs
# -----------------------
def run_scrape_from_csv_posts() -> None:

    OUT_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(INPUT_CSV)
    id_col = 'user_id'

    logging.info("Loaded %d rows from %s (user id column: %s)", len(df), INPUT_CSV, id_col)

    today = yyyymmdd_utc()

    for i, row in df.iterrows():
        user_id_raw = row[id_col]
        if pd.isna(user_id_raw):
            continue

        user_id = str(user_id_raw).strip()
        if not user_id or user_id.lower() == "nan":
            continue

        logging.info("[%d/%d] Scraping posts for user_id=%s", i + 1, len(df), user_id)

        cursor = None
        page = 1

        while page <= MAX_PAGES_PER_USER:
            fname = f"{safe_filename(user_id)}_{today}_page{page:03d}.json"
            out_path = OUT_DIR / fname

            if out_path.exists() and not OVERWRITE_EXISTING:
                logging.info("  page %03d exists, skipping: %s", page, fname)

                # If you skip existing pages, you might still want to continue pagination;
                # but without reading the existing file to get cursor, we can't.
                # So safest behavior: stop at first existing page (common in incremental scrapes).
                break

            try:
                raw_payload = scrape_user_posts(
                    user_id=user_id,
                    token=TOKEN,
                    chunk_size=CHUNK_SIZE,
                    cursor=cursor,
                )

                next_cursor = find_next_cursor(raw_payload)

                wrapper = {
                    "_meta": {
                        "scrape_ts": utc_now_iso(),
                        "endpoint": ENDPOINT,
                        "query_id": user_id,
                        "page": page,
                        "chunk_size": CHUNK_SIZE,
                        "cursor_used": cursor,
                        "cursor_next": next_cursor,
                        "request_params": {"id": user_id, "chunk_size": CHUNK_SIZE, "cursor": cursor},
                    },
                    "data": raw_payload,
                }

                out_path.write_text(json.dumps(wrapper, ensure_ascii=False, indent=2), encoding="utf-8")
                logging.info("  saved: %s (next_cursor=%s)", fname, bool(next_cursor))

                time.sleep(SLEEP_SECS)

                # Stop condition: no cursor returned (no more pages)
                if not next_cursor:
                    break

                cursor = next_cursor
                page += 1

            except requests.HTTPError as e:
                err_wrapper = {
                    "_meta": {
                        "scrape_ts": utc_now_iso(),
                        "endpoint": ENDPOINT,
                        "query_id": user_id,
                        "page": page,
                        "chunk_size": CHUNK_SIZE,
                        "cursor_used": cursor,
                        "error": f"HTTPError: {e}",
                    },
                    "data": None,
                }
                out_path.write_text(json.dumps(err_wrapper, ensure_ascii=False, indent=2), encoding="utf-8")
                logging.exception("HTTP error for user_id=%s page=%s", user_id, page)
                break

            except Exception as e:
                err_wrapper = {
                    "_meta": {
                        "scrape_ts": utc_now_iso(),
                        "endpoint": ENDPOINT,
                        "query_id": user_id,
                        "page": page,
                        "chunk_size": CHUNK_SIZE,
                        "cursor_used": cursor,
                        "error": f"Exception: {e}",
                    },
                    "data": None,
                }
                out_path.write_text(json.dumps(err_wrapper, ensure_ascii=False, indent=2), encoding="utf-8")
                logging.exception("Unexpected error for user_id=%s page=%s", user_id, page)
                break

In [8]:
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

    # 1) Scrape and save raw JSONs for posts
    run_scrape_from_csv_posts()

INFO: Loaded 1 rows from Threads/user_info.csv (user id column: user_id)
INFO: [1/1] Scraping posts for user_id=13460080
ERROR: HTTP error for user_id=13460080 page=1
Traceback (most recent call last):
  File "/var/folders/z0/01jh3sgn3wq18wjljcjkl1x40000gn/T/ipykernel_69112/1121599699.py", line 42, in run_scrape_from_csv_posts
    raw_payload = scrape_user_posts(
                  ^^^^^^^^^^^^^^^^^^
  File "/var/folders/z0/01jh3sgn3wq18wjljcjkl1x40000gn/T/ipykernel_69112/2187952338.py", line 63, in scrape_user_posts
    res.raise_for_status()
  File "/Users/pouriakhansari/anaconda3/lib/python3.11/site-packages/requests/models.py", line 1021, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 493 Client Error:  for url: https://ensembledata.com/apis/threads/user/posts?id=13460080&chunk_size=5&token=GobJlP7vKjn4Z5Jw


In [22]:
# -----------------------
# OPTIONAL Step 2: Parse saved JSONs into a posts table
# -------------------
# ----


HASHTAG_RE = re.compile(r"#(\w+)")
MENTION_RE = re.compile(r"@(\w+)")


def extract_hashtags(text: str) -> str:
    return ", ".join(HASHTAG_RE.findall(text)) if text else ""


def extract_mentions(text: str) -> str:
    return ", ".join(MENTION_RE.findall(text)) if text else ""

from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

def best_image_candidate(obj: Dict[str, Any]) -> Tuple[str, Optional[int], Optional[int]]:
    """
    Return (url, width, height) from the best available image candidate.
    Works for:
      - media dicts inside carousel_media (have image_versions2.candidates)
      - post dict itself (may have image_versions2.candidates)
    Preference: largest area candidate.
    """
    candidates = safe_get(obj, ["image_versions2", "candidates"], []) or []
    if not isinstance(candidates, list) or not candidates:
        return "", None, None

    best = None
    best_area = -1

    for c in candidates:
        if not isinstance(c, dict):
            continue
        url = c.get("url")
        w = c.get("width")
        h = c.get("height")
        if not url:
            continue
        if isinstance(w, int) and isinstance(h, int) and w > 0 and h > 0:
            area = w * h
        else:
            area = 0

        if area > best_area:
            best = c
            best_area = area

    if not best:
        # fallback to first url
        first = candidates[0]
        return first.get("url", "") or "", first.get("width"), first.get("height")

    return best.get("url", "") or "", best.get("width"), best.get("height")


def extract_posts_from_payload(raw_payload: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Parse /threads/user/posts response into a flat list of post rows.

    Input: the *API JSON response dict* (NOT your wrapper).
    Your response format: {"data": [ { "node": { "thread_items": [ {"post": {...}}, ...] } , "cursor": ... }, ... ] }
    """
    rows: List[Dict[str, Any]] = []

    edges = raw_payload.get("mediaData",{}).get("edges",{})
    #print(edges)
    if not isinstance(edges, list):
        return rows

    for edge in edges:
        if not isinstance(edge, dict):
            continue

        node = edge.get("node", {}) or {}
        if not isinstance(node, dict):
            continue

        thread_items = node.get("thread_items", []) or []
        if not isinstance(thread_items, list):
            continue

        for item in thread_items:
            if not isinstance(item, dict):
                continue

            # ✅ Correct: post is under "post"
            post = item.get("post", {}) or {}
            if not isinstance(post, dict):
                continue

            # ----------------------------
            # USER DETAILS
            # ----------------------------
            user = post.get("user", {}) or {}
            user_id = user.get("id") or user.get("pk") or ""
            username = user.get("username", "") or ""
            profile_pic_url = user.get("profile_pic_url", "") or ""
            full_name=user.get("full_name", "") or ""
            is_verified = bool(user.get("is_verified", False))

            # ----------------------------
            # POST IDS + TIMESTAMP
            # ----------------------------
            threads_post_id = str(post.get("id", "") or "")
            threads_pk = str(post.get("pk", "") or "")
            
            taken_at = post.get("taken_at", None)
            readable_date = ""
            if isinstance(taken_at, (int, float)) and taken_at:
                readable_date = datetime.fromtimestamp(int(taken_at)).strftime("%Y-%m-%d %H:%M:%S")

            # ----------------------------
            # TEXT (fragments) + CAPTION
            # ----------------------------
            fragments = safe_get(post, ["text_post_app_info", "text_fragments", "fragments"], []) or []
            fragments_text = ""
            if isinstance(fragments, list) and fragments:
                parts = []
                for f in fragments:
                    if isinstance(f, dict):
                        t = f.get("plaintext", "")
                        if t:
                            parts.append(t)
                fragments_text = "".join(parts).strip()

            caption_text = safe_get(post, ["caption", "text"], "") or ""
            hashtags = extract_hashtags(caption_text)
            mentions = extract_mentions(caption_text)
            is_paid_partnership = post.get("is_paid_partnership", False)
            has_audio = post.get("has_audio")or ""
            media_type = post.get("media_type")or ""
            like_and_view_counts_disabled = post.get("like_and_view_counts_disabled", False) 
            is_reply = post.get("text_post_app_info").get("is_reply") 
            is_pinned_to_profile = (post.get("text_post_app_info", {})
                                  .get("pinned_post_info", {})
                                  .get("is_pinned_to_profile", False)
)
            # ----------------------------
            # ENGAGEMENT
            # ----------------------------
            like_count = post.get("like_count", 0) or 0
            reshare_count = safe_get(post, ["text_post_app_info", "reshare_count"], 0) or 0
            direct_reply_count = safe_get(post, ["text_post_app_info", "direct_reply_count"], 0) or 0
            repost_count = safe_get(post, ["text_post_app_info", "repost_count"], 0) or 0
            quote_count = safe_get(post, ["text_post_app_info", "quote_count"], 0) or 0



            original_width = post.get("original_width", False)
            original_height = post.get("original_height",False)

            # ----------------------------
            # MEDIA (carousel vs single)
            # ----------------------------
            carousel_media = post.get("carousel_media", []) or []
            image_count = 0
            carousel_media_urls: List[str] = []

            cover_url, cover_width, cover_height = "", None, None

            if carousel_media and isinstance(carousel_media, list) and media_type != 2:
                image_count = len(carousel_media)
                cover_url, cover_width, cover_height = best_image_candidate(carousel_media[0])
                for m in carousel_media:
                    if isinstance(m, dict):
                        u, _, _ = best_image_candidate(m)
                        if u:
                            carousel_media_urls.append(u)
            elif media_type != 2:
                # single image (not video)
                cover_url, cover_width, cover_height = best_image_candidate(post)
                image_count = 1 if cover_url else 0
                if cover_url:
                    carousel_media_urls.append(cover_url)
            else:
                # video (media_type == 2)
                image_count = 0
                cover_url, cover_width, cover_height = best_image_candidate(post)
            
            video_versions = post.get("video_versions") or []
            video_urls = {v.get("type"): v.get("url") for v in video_versions}
            hd_url = video_urls.get(103, "")
            sd_url = video_urls.get(101, "")
            
            
            rows.append({
                "threads_post_id": threads_post_id,
                "threads_pk": threads_pk,
                "user_id": user_id,
                "username": username,
                # "profile_pic_url": profile_pic_url,  # keep if you want
                "full_name": full_name,
                "is_verified": is_verified,
                "Timestamp": readable_date,
                "is_paid_partnership": is_paid_partnership,
                "is_reply": is_reply,
                "like_and_view_counts_disabled": like_and_view_counts_disabled,
                "is_pinned": is_pinned_to_profile,

                # "text_fragments_text": fragments_text,  # keep if you want
                "caption_text": caption_text,
                "hashtags": hashtags,
                "mentions": mentions,

                "like_count": like_count,
                "reshare_count": reshare_count,
                "comment_count": direct_reply_count,
                "repost_count": repost_count,
                "quote_count": quote_count,  # keep if you want
                
                "media_type": media_type,
                "image_count": image_count,
                "has_audio": has_audio,
                "original_width": original_width,
                "original_height": original_height,
                "cover_url": cover_url,
                "cover_width": cover_width,
                "cover_height": cover_height,

                "carousel_media_urls": ", ".join(carousel_media_urls),
                "video_hd_url": hd_url,
                "video_sd_url": sd_url
            })

    return rows


def parse_all_posts_jsons_to_csv(out_csv: Path = Path("Threads") / "post_info.csv", encoding="utf-8-sig") -> pd.DataFrame:
    if not OUT_DIR.exists():
        raise FileNotFoundError(f"Folder not found: {OUT_DIR}. Run scraping first.")

    rows: List[Dict[str, Any]] = []

    for fp in sorted(OUT_DIR.glob("*.json")):
        wrapper = json.loads(fp.read_text(encoding="utf-8"))
        print(fp)
        meta = wrapper.get("_meta", {}) if isinstance(wrapper, dict) else {}
        scrape_ts = meta.get("scrape_ts")
        query_id = meta.get("query_id")
        page = meta.get("page")

        raw_payload = wrapper.get("data") if isinstance(wrapper, dict) else None
        if not isinstance(raw_payload, dict):
            continue

        #payload = raw_payload.get("data")
        payload = raw_payload.get("mediaData", {})
        #print(payload)
        if isinstance(payload, dict):
            # NOTE: wrapper["data"] is raw_payload; raw_payload itself should be the API response dict
            api_payload = raw_payload
        else:
            api_payload = raw_payload

        posts = extract_posts_from_payload(api_payload)
        for p in posts:
            p["scrape_ts"] = scrape_ts
            p["query_id"] = query_id
            p["page"] = page
            p["source_file"] = fp.name
            rows.append(p)

    df_out = pd.DataFrame(rows)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(out_csv, index=False)
    logging.info("Saved parsed posts CSV: %s (%d rows)", out_csv, len(df_out))
    return df_out

In [23]:
# -----------------------
# Main
# -----------------------
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")


    # 2) Optional: parse into a flat table
    df=parse_all_posts_jsons_to_csv()


INFO: Saved parsed posts CSV: Threads/post_info.csv (149 rows)


Threads/post_info/nike_all_pages.json
Threads/post_info/nike_page_1.json
Threads/post_info/nike_page_10.json
Threads/post_info/nike_page_11.json
Threads/post_info/nike_page_12.json
Threads/post_info/nike_page_13.json
Threads/post_info/nike_page_2.json
Threads/post_info/nike_page_3.json
Threads/post_info/nike_page_4.json
Threads/post_info/nike_page_5.json
Threads/post_info/nike_page_6.json
Threads/post_info/nike_page_7.json
Threads/post_info/nike_page_8.json
Threads/post_info/nike_page_9.json


In [17]:
df.head()

,threads_post_id,threads_pk,user_id,username,full_name,is_verified,Timestamp,is_paid_partnership,is_reply,like_and_view_counts_disabled,...,cover_url,cover_width,cover_height,carousel_media_urls,video_hd_url,video_sd_url,scrape_ts,query_id,page,source_file
0,3836254256739690476_63454848350,3836254256739690476,63454848350,nike,Nike,True,2026-02-19 16:39:26,None,False,False,...,https://instagram.fyyz1-2.fna.fbcdn.net/v/t51....,1440.0,1800.0,https://instagram.fyyz1-2.fna.fbcdn.net/v/t51....,,,None,None,None,nike_page_1.json
1,3835523584857660664_63454848350,3835523584857660664,63454848350,nike,Nike,True,2026-02-18 16:27:43,None,False,False,...,https://instagram.fyyz1-2.fna.fbcdn.net/v/t51....,1440.0,1801.0,https://instagram.fyyz1-2.fna.fbcdn.net/v/t51....,,,None,None,None,nike_page_1.json
2,3829507873022197862_63454848350,3829507873022197862,63454848350,nike,Nike,True,2026-02-10 09:15:34,None,False,False,...,https://instagram.fyyz1-2.fna.fbcdn.net/v/t51....,1440.0,1800.0,https://instagram.fyyz1-2.fna.fbcdn.net/v/t51....,,,None,None,None,nike_page_1.json
3,3828928273242473482_63454848350,3828928273242473482,63454848350,nike,Nike,True,2026-02-09 14:04:00,None,False,False,...,https://instagram.fyyz1-2.fna.fbcdn.net/v/t51....,1440.0,1800.0,https://instagram.fyyz1-2.fna.fbcdn.net/v/t51....,,,None,None,None,nike_page_1.json
4,3828458202728602651_63454848350,3828458202728602651,63454848350,nike,Nike,True,2026-02-08 22:30:22,None,False,False,...,https://instagram.fyyz1-1.fna.fbcdn.net/v/t51....,640.0,800.0,,https://instagram.fyyz1-2.fna.fbcdn.net/o1/v/t...,https://instagram.fyyz1-2.fna.fbcdn.net/o1/v/t...,None,None,None,nike_page_1.json


In [52]:
df.to_csv()

,threads_post_id,threads_pk,user_id,username,full_name,is_verified,Timestamp,like_and_view_counts_disabled,caption_text,hashtags,...,carousel_media_urls,is_paid_partnership,has_audio,media_type,hd_url,sd_url,scrape_ts,query_id,page,source_file
7,3591110821369513527_63454848350,3591110821369513527,63454848350,nike,Nike,True,2025-03-18 12:04:01,,Queens don’t ask for permission. \nNobody rule...,,...,,,True,2,https://scontent-yyz1-1.cdninstagram.com/o1/v/...,https://scontent-yyz1-1.cdninstagram.com/o1/v/...,None,None,None,nike_page_1.json


In [87]:
df[df["media_type"]==19]

,threads_post_id,threads_pk,user_id,username,full_name,is_verified,Timestamp,is_paid_partnership,is_reply,like_and_view_counts_disabled,...,cover_url,cover_width,cover_height,carousel_media_urls,video_hd_url,video_sd_url,scrape_ts,query_id,page,source_file
25,3564574447316686362_63454848350,3564574447316686362,63454848350,nike,Nike,True,2025-02-09 20:20:09,None,False,False,...,,NaN,NaN,,,,None,None,None,nike_page_3.json
26,3564574476852801459_63454848350,3564574476852801459,63454848350,nike,Nike,True,2025-02-09 20:20:12,None,True,False,...,,NaN,NaN,,,,None,None,None,nike_page_3.json
31,3541230777931531602_63454848350,3541230777931531602,63454848350,nike,Nike,True,2025-01-08 15:20:26,None,True,False,...,,NaN,NaN,,,,None,None,None,nike_page_3.json
53,3404083317646519971_63454848350,3404083317646519971,63454848350,nike,Nike,True,2024-07-03 10:52:55,None,True,False,...,,NaN,NaN,,,,None,None,None,nike_page_5.json
